<a href="https://colab.research.google.com/github/Aper777/AI/blob/main/%D0%9A%D0%BE%D0%BF%D0%B8%D1%8F_%D0%B1%D0%BB%D0%BE%D0%BA%D0%BD%D0%BE%D1%82%D0%B0_%22H2_KNN_ipynb%22.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Homework: k-Nearest Neighbors Classifier (Multiple Distance Metrics)

In this homework, you will implement the missing parts of the `KNearestNeighbor` class to create a kNN classifier that can use **different distance metrics**.

## Tasks
1. Implement `compute_distances` so it supports:
    - **Hamming Distance**
    - **Euclidean Distance**
    - **Manhattan Distance**
2. Implement `predict_labels` to choose the majority label among the `k` nearest neighbors.
3. Test your implementation on sample data with different distance metrics.



In [ ]:
import numpy as np
import pandas as pd


In [ ]:
from typing import Counter
class KNearestNeighbor(object):
    """a kNN classifier with multiple distance metrics"""

    def __init__(self, X_train, y_train):
        """
        Initializing the KNN object
        """
        self.X_train = X_train
        self.y_train = y_train

    def fit_predict(self, X_test, k=1, distance="hamming"):
        """
        Fits the model and predicts labels for test data.

        Parameters:
        - X_test: Test data
        - k: Number of nearest neighbors
        - distance: Distance metric ('hamming', 'euclidean', 'manhattan')
        """
        dists = self.compute_distances(X_test, distance)
        return self.predict_labels(dists, k=k)

    def compute_distances(self, X_test, distance="hamming"):
        """
        Compute the distance between each test and train point.
        Supported distances: 'hamming', 'euclidean', 'manhattan'
        """

        # Ստեղծում ենք դատարկ ցուցակ (list), որտեղ կպահենք բոլոր test նմուշների հեռավորությունները
        dists = []
        # Անցնում ենք test dataset-ի յուրաքանչյուր sample-ի (տողի) վրայով
        for X in X_test:

          if distance== "hamming":
            # Հաշվում ենք միջին բացարձակ տարբերությունը յուրաքանչյուր train նմուշի և X-ի միջև
            # np.abs(...) → վերցնում է բացարձակ տարբերությունը
            # np.mean(..., axis=1) → հաշվում է միջինը յուրաքանչյուր տողի համար
            dist = np.mean(np.abs(self.X_train - X), axis = 1)
          elif distance == "euclidean":
            # Հաշվում ենք √((x₁−y₁)² + (x₂−y₂)² + ... + (xₙ−yₙ)²)
            # (self.X_train - X)**2 → տարբերությունների քառակուսին
            # np.sum(..., axis=1) → գումարում է բոլոր հատկանիշները յուրաքանչյուր տողի համար
            # np.sqrt(...) → քառակուսի արմատը՝ ստանալու վերջնական Euclidean հեռավորությունը
            dist = np.sqrt(np.sum((self.X_train - X)**2, axis=1))
          elif distance == "manhattan":
            # Հաշվում ենք բացարձակ տարբերությունների գումարը
            # np.abs(...) → բացարձակ արժեք
            # np.sum(..., axis=1) → գումարում է հատկանիշներով (յուրաքանչյուր տողի համար)
            dist = np.sum(np.abs(self.X_train - X), axis=1)
          else:
            # Բարձրացնում է սխալ՝ հստակ նշելով պատճառը
            raise ValueError("Unsupported distance metric.")

          # Ավելացնում ենք հաշվված հեռավորությունների զանգվածը dists ցուցակում
          # dist-ը պարունակում է յուրաքանչյուր train նմուշի հեռավորությունը տվյալ X test նմուշից
          dists.append(dist)

        return np.array(dists)


    def predict_labels(self, dists, k=1):
        """
        Predict labels based on nearest neighbors.
        """

        # Ստեղծում ենք դատարկ ցուցակ (list), որտեղ կպահենք բոլոր test նմուշների կանխատեսված պիտակները (labels)։
        y_pred = []

        for dist in dists:
          # Անցնում ենք յուրաքանչյուր test նմուշի հեռավորությունների տողի վրայով։
          # Այսինքն՝ dist պարունակում է մեկ test նմուշի բոլոր train նմուշների հեռավորությունները։
          # Օրինակ՝ եթե ունենք 5 train նմուշ, ապա dist = [0.3, 0.8, 0.1, 1.0, 0.6]։
          nearest_idx = np.argsort(dist)[:k]
          # np.argsort(dist) վերադարձնում է train նմուշների ինդեքսները աճման կարգով՝ ըստ հեռավորության։
          # Այսինքն՝ ամենափոքր արժեքները՝ առաջինը (ամենամոտները)։
          # Օրինակ՝
          # Եթե dist = [0.8, 0.1, 0.5], ապա np.argsort(dist) → [1, 2, 0]
          # և [:k] վերցնում է առաջին k ինդեքսները (օր.՝ [1, 2], եթե k=2)։

          # Այսպիսով, nearest_idx → ամենամոտ k train նմուշների ինդեքսներն են։
          nearest_label = self.y_train[nearest_idx]
          # Այս տողը վերցնում է այն train նմուշների պիտակները (labels), որոնց ինդեքսները nearest_idx-ում են։
          # Օրինակ՝ եթե nearest_idx = [1, 2, 4] և
          # self.y_train = [0, 1, 1, 2, 0]
          # ապա nearest_label = [1, 1, 0]։

          # Այսինքն՝ մոտակա հարևանների պիտակների ցանկ։
          most_comon = Counter(nearest_label).most_common(1)[0][0]
          # Այստեղ որոշում ենք, թե որ պիտակն է ամենաշատը հանդիպում հարևանների մեջ։

          # Counter(nearest_label) → հաշվում է, թե քանի անգամ է յուրաքանչյուր label հանդիպում։
          # Օրինակ՝ Counter([1, 1, 0]) = {1: 2, 0: 1}

          # .most_common(1) → վերադարձնում է list՝ ամենահաճախ հանդիպող արժեքը և դրա քանակը։
          # Օր.՝ [(1, 2)]

          # [0][0] → վերցնում ենք հենց label-ի արժեքը (այստեղ՝ 1)։

          # Այսինքն՝ most_comon = այն դասը (label-ը), որն առավել հաճախ է հանդիպել k մոտակա նմուշների մեջ։
          y_pred.append(most_comon)

        return y_pred



**Example Dataset**

In [ ]:
# Training data
X_train = np.array([
    [1, 0, 1],
    [0, 1, 0],
    [1, 1, 1],
    [0, 0, 0]
])
y_train = np.array([0, 1, 0, 1])

# Test data
X_test = np.array([
    [1, 0, 0],
    [0, 1, 1]
])


**Test the Model**

In [ ]:
knn = KNearestNeighbor(X_train, y_train)

for dist in ["hamming", "euclidean", "manhattan"]:
    y_pred = knn.fit_predict(X_test, k=3, distance=dist)
    print(f"Distance: {dist} -> Predictions: {y_pred}")
